# Aula 04 – Primeira Rede Neural (MLP) com Keras
## Disciplina: Redes Neurais Artificiais, Deep Learning e Algoritmos Genéticos – 1TIAPG

**Objetivos da aula:**
- Relembrar Perceptron e entender por que precisamos de várias camadas (MLP).
- Montar uma rede neural feedforward simples usando Keras (TensorFlow).
- Treinar e avaliar um modelo de classificação binária (cliente compra ou não compra).
- Fazer pequenos experimentos com número de neurônios e épocas.

**Contexto:**
Você é analista de dados em um e-commerce de São Paulo. Queremos prever se um cliente vai **comprar (1) ou não comprar (0)** com base no seu comportamento no site.

2. Imports e setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set(style="whitegrid")
print("TensorFlow:", tf.__version__)

3. Geração de dados sintéticos “realistas”

“Este dataset é sintético, gerado para simular um cenário realista.”

In [ ]:
# Gerando um dataset sintético de classificação binária
# Cada linha = um visitante do site
# y = 1 se comprou, 0 se não comprou

X, y = make_classification(
    n_samples=2000,
    n_features=5,
    n_informative=4,
    n_redundant=0,
    n_clusters_per_class=2,
    random_state=42
)

df = pd.DataFrame(X, columns=[
    "tempo_no_site",          # em minutos
    "paginas_visitadas",      # quantidade de páginas
    "valor_medio_carrinho",   # valor médio dos produtos vistos
    "veio_de_campanha",       # 1 se clicou em campanha/ads, 0 se veio direto
    "dispositivo_mobile"      # 1 se mobile, 0 se desktop
])
df["comprou"] = y

df.head()

4. EDA mínima

In [ ]:
df.describe()

In [ ]:
# Distribuição da variável alvo
sns.countplot(x="comprou", data=df)
plt.title("Distribuição da variável alvo (comprou)")
plt.show()

In [ ]:
# Amostra para não ficar muito pesado
sns.pairplot(df.sample(300, random_state=42), hue="comprou")
plt.show()

DICA IMPORTANTE: “Antes de treinar rede neural, sempre olhar a base.”

5. Separar treino e teste + normalização

In [ ]:
X = df.drop("comprou", axis=1).values
y = df["comprou"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape

DICA: “Redes neurais treinam melhor quando os dados são escalonados (média ~0, desvio ~1).”

6. Definição da MLP (modelo base)

In [ ]:
# Definindo uma rede neural feedforward (MLP) simples

model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),  # 5 features
    layers.Dense(8, activation="relu"),              # camada escondida
    layers.Dense(4, activation="relu"),              # outra camada escondida
    layers.Dense(1, activation="sigmoid")            # saída binária (probabilidade)
])

model.summary()

IMPORTANTE:

- “Cada Dense é uma camada totalmente conectada (todos neurônios conectados com a camada anterior).”
- “relu nas camadas escondidas → boa para aprender não linearidades.”
- “sigmoid na saída → probabilidade de comprou = 1.”

7. Compilar o modelo

In [ ]:
model.compile(
    optimizer="adam",                # variante do gradiente descendente
    loss="binary_crossentropy",      # erro para classificação binária
    metrics=["accuracy"]             # vamos acompanhar accuracy
)

COMENTÁRIO:

“Aqui o Keras está configurando como vai fazer o feedforward + backprop internamente.”

8. Treinar o modelo

In [ ]:
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,  # parte do treino vira validação
    epochs=20,
    batch_size=32,
    verbose=0              # mude para 1 se quiser ver o log em tempo real
)

9. Curvas de treinamento (loss e accuracy)

In [ ]:
# Curva de Loss
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history["loss"], label="Treino")
plt.plot(history.history["val_loss"], label="Validação")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.title("Curva de Loss")

# Curva de Accuracy
plt.subplot(1,2,2)
plt.plot(history.history["accuracy"], label="Treino")
plt.plot(history.history["val_accuracy"], label="Validação")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Curva de Accuracy")

plt.tight_layout()
plt.show()

PERGUNTAS PARA DEBATE:

- “As curvas de treino e validação estão parecidas ou se afastando?”
- “Parece que tem overfitting? Em que época ele começaria?”

10. Avaliação no conjunto de teste

In [ ]:
# Predição no conjunto de teste
y_pred_proba = model.predict(X_test_scaled).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

print("Accuracy no teste:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de confusão")
plt.show()

Desafios guiados 

### Desafio 1 – Aumente a capacidade da rede

Tarefa:
- Aumente o número de neurônios e/ou o número de camadas escondidas.
- Exemplo: 16 neurônios na primeira camada, 8 na segunda, 4 na terceira.
- Re-treine o modelo por 20 épocas e compare:
  - Accuracy no treino e validação.
  - Accuracy no teste.
  - Curvas de loss e accuracy.

Perguntas:
- O modelo melhorou?
- Apareceu overfitting?

Código-esqueleto para mexer e reaproveitar as funções de plot + avaliação.

In [ ]:
model2 = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(4, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model2.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history2 = model2.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=0
)

Desafio 2 – Mudar o número de epochs

### Desafio 2 – Menos ou mais épocas?

Tarefa:
- Treine a mesma arquitetura com:
  - 5 épocas
  - 50 épocas
- Compare:
  - Tempo de treino
  - Curvas de treino/validação
  - Accuracy no teste

Perguntas:
- Em 5 épocas, o modelo já aprendeu o suficiente?
- Em 50 épocas, acontece overfitting?

Desafio 3 – Trocar função de ativação

### Desafio 3 – E se trocarmos a função de ativação?

Tarefa:
- Troque `activation="relu"` por `activation="tanh"` nas camadas escondidas.
- Mantenha o resto igual.
- Compare desempenho com a versão original.

Perguntas:
- Qual ativação funcionou melhor?
- Você observa diferenças nas curvas de loss/accuracy?